## Libraries

In [0]:
%pip install xgboost==2.0.3


In [0]:
import os
import mlflow
import mlflow.spark

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import xgboost as xgb

In [0]:
data = spark.read.table("workspace.telco.ml_silver_data")

In [0]:
data = data.drop("customerID")

In [0]:
display(data.limit(5))

In [0]:
feature_cols = [c for c in data.columns if c != "Churn"]


In [0]:
data = data.withColumn("label", data["Churn"].cast("double"))


## Modeling

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [0]:
assembled_data = assembler.transform(data).select("features", "label")

In [0]:
train, test = assembled_data.randomSplit([0.8, 0.2], seed=42)

In [0]:
train_df = train.select("features", "label").toPandas()

dtrain = xgb.DMatrix(train_df["features"].tolist(), label=train_df["label"])

params = {
    "objective": "binary:logistic",
    "eta": 0.1,
    "max_depth": 6,
}

xgb_model = xgb.train(params, dtrain, num_boost_round=100)


In [0]:
# 4. Convert train to pandas
train_df = train.toPandas()

# 5. Prepare XGBoost input
import xgboost as xgb
dtrain = xgb.DMatrix(
    data = train_df["features"].apply(lambda v: v.toArray()).tolist(),
    label = train_df["label"]
)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr_model = LogisticRegression(labelCol="label", featuresCol="features")

from pyspark.ml.classification import RandomForestClassifier

rf_model = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=200)

from pyspark.ml.classification import GBTClassifier

gb_model = GBTClassifier(labelCol="label", featuresCol="features", maxIter=50)


In [0]:
fitted_lr_model = lr_model.fit(train)
fitted_rf_model = rf_model.fit(train)
fitted_gb_model = gb_model.fit(train)

params = {
    "objective": "binary:logistic",
    "eta": 0.1,
    "max_depth": 6,
}

fitted_xgb_model = xgb.train(params, dtrain, num_boost_round=100)

In [0]:
test_df = test.toPandas()

# Convert vectors → arrays for XGBoost
X_test = test_df["features"].apply(lambda v: v.toArray()).tolist()
y_test = test_df["label"]

# Create DMatrix
dtest = xgb.DMatrix(X_test, label=y_test)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import Row
from pyspark.ml.linalg import Vectors

lr_predictions = fitted_lr_model.transform(test)
rf_predictions = fitted_rf_model.transform(test)
gb_predictions = fitted_gb_model.transform(test)
xgb_pred_proba = fitted_xgb_model.predict(dtest)
xgb_pred_label = (xgb_pred_proba >= 0.5).astype(int)

# Build Spark rows
rows = []
for prob, label, pred in zip(xgb_pred_proba, y_test, xgb_pred_label):
    rows.append(Row(
        label=float(label),
        prediction=float(pred),
        probability=Vectors.dense([1 - prob, prob])
    ))

# Create Spark DataFrame
xgb_predictions_spark = spark.createDataFrame(rows)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

lr_roc_auc = evaluator.evaluate(lr_predictions)
print("LogisticRegression-ROC-AUC:", lr_roc_auc)
rf_roc_auc = evaluator.evaluate(rf_predictions)
print("RandomForest-ROC-AUC:", rf_roc_auc)
gb_roc_auc = evaluator.evaluate(gb_predictions)
print("GradientBoosting-ROC-AUC:", gb_roc_auc)
xgb_roc_auc = evaluator.evaluate(xgb_predictions_spark)
print("XGB-ROC-AUC:", xgb_roc_auc)

